# Training: Bank Marketing

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "../.."))

In [ ]:
from src.data import load_dataloaders_from_hf
from src.model import get_model
from src.config import get_dataset_config
from src.train import train
from src.evaluate import evaluate

## Config

In [ ]:
# Load dataset configuration (unified task-driven architecture)
DATASET_NAME = "bank-marketing"
config = get_dataset_config(DATASET_NAME)

CONFIG = {
    "dataset": DATASET_NAME,
    "task": config.task,
    "model": config.model_config.get("type", "linear"),
    "batch_size": 64,
    "epochs": 100,
    "lr": 5e-3,
}

## Load Data

In [ ]:
train_loader, test_loader, INPUT_DIM = load_dataloaders_from_hf(
    dataset_name=config.hf_repo,
    batch_size=CONFIG["batch_size"],
    label_col=config.label_col,
)

## Model

In [ ]:
# Create model using the unified factory function
model = get_model(INPUT_DIM, CONFIG["task"], config.model_config)

## Training 

In [ ]:
run = train(
    model,
    train_loader,
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    task=CONFIG["task"],
    use_wandb=True,
    config=CONFIG,
    model_name=CONFIG["model"],
)

## Evaluation on Test Set

In [ ]:
metrics, preds, targets = evaluate(
    model, test_loader, task=CONFIG["task"], use_wandb=True, run=run
)

In [ ]:
import numpy as np
from src.visualize import plot_confusion_matrix, plot_roc_curve

# Print metrics
print("\n" + "=" * 50)
print("CLASSIFICATION METRICS")
print("=" * 50)
for metric_name, metric_value in metrics.items():
    print(f"{metric_name:.<20} {metric_value:.4f}")
print("=" * 50 + "\n")

# Plot confusion matrix
plot_confusion_matrix(targets, preds, title="Bank Marketing - Confusion Matrix")

# Compute probabilities for ROC curve (convert binary predictions to soft probabilities)
y_probs = np.array(preds)
plot_roc_curve(targets, y_probs, title="Bank Marketing - ROC Curve")

## Save Model Artifacts

In [ ]:
from src import save

models_base_dir = os.path.abspath(os.path.join(os.getcwd(), "../..", "models"))

save.save_model_artifact(
    model,
    dataset_name="bank-marketing",
    filename="centralized_lr_bank_model.pt",
    run=run,
    models_dir=models_base_dir,
)

print(f"Model saved to {models_base_dir}")

In [ ]:
# Finish W&B run
if run is not None:
    run.finish()